In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Image2Sphere (Klee et al., ICLR 2023; --model i2s_real, the published model under this harness) on SYMSOL I
# with 10 % of the training data: 4,500 of 45,000 views per shape (tet, cube, icosa, cone, cyl), one model for
# all five, as the symsol1_10pct_* runs of this repo. The paper reports only the log-likelihood on SYMSOL
# (Table 3: 2.98 with 10k views per shape, 3.41 with 50k); here every epoch also scores the MEDIAN ROTATION
# ERROR to the nearest equivalent pose (datasets/symsol.py, engine/metrics.py nearest_equivalent_error) on
# the first 1,000 test views per shape (as the flow runs), and at the end the per-shape medians, Acc@15/30 and the grid
# log-likelihood read off the same histogram upstream's evaluate_ll uses (engine/grid_model_likelihood.py).
#
# Two recipes:
#   i2s_symsol1_10pct_sgd   the paper's: ResNet-50 (ImageNet), SGD + Nesterov 0.9, lr 1e-3, x0.1 every 15 epochs,
#                           batch 64, 40 epochs, no EMA, no augmentation, argmax on the level-5 grid (1.875 deg).
#   i2s_symsol1_10pct_adamw the schedule of the symsol1_10pct_*_i2s_adamw flow runs it is compared with:
#                           AdamW, lr 1e-4 x sqrt(64/32), 5 warmup epochs + cosine, batch 64, 40 epochs, no EMA.
# The level-5 grid is a 4.3 GB Wigner matrix built on the GPU: fine on an RTX 6000 / L4 / T4 (16 GB); use
# --i2s_eval_rec_level 4 (3.75 deg, 540 MB) on a smaller card.
#
# Attach the Kaggle dataset syfry5suvzovvakmuj/symsol1-10pct-fulltest (symsol/{train,test}/rotations.npz +
# images/). With internet (T4): the branch below is cloned and the dependencies installed from PyPI/GitHub,
# W&B syncs live from the `wandb_api_key` secret (or WANDB_API_KEY below). Without internet (RTX): also attach
# ONE repo snapshot (a GA-Research/ tree or ga-research-*.tar.gz) and ga-research-offline-deps; wandb runs
# offline and gpu_pool/rtx_monitor.py replays the WANDB_SYNC lines into W&B.
WANDB_API_KEY = None
REPO = 'chagrygoris/GA-Research'
BRANCH = 'claude/optimistic-faraday-6effbq'
# One process (--no-ddp): the loader keeps the 47,500 decoded images in RAM (7 GB) and a DDP rank would hold a copy
# each, which with the decoding threads exceeds a Kaggle T4 x2 kernel's 29 GB and gets a rank SIGKILLed before
# the first epoch. Batch 64 on one T4 is the paper's setting. --symsol_test_views 1000 scores the same first
# 1,000 test views per shape as the symsol1_10pct_* flow runs.
COMMON = ['--model', 'i2s_real', '--encoder', 'resnet50', '--dataset', 'symsol', '--symsol_set', '1',
          '--symsol_train_views', '4500', '--symsol_test_views', '1000', '--no-use_warp', '--no-use_synth', '--no-ema',
          '--no-fixed_val_noise', '--no-ddp', '--batch_size', '64', '--n_epochs', '40', '--i2s_eval_rec_level', '5',
          '--num_workers', '4']
VARIANTS = {
    'i2s_symsol1_10pct_sgd': ['--optimizer', 'sgd', '--lr', '1e-3', '--lr_scaling', 'none', '--lr_schedule', 'step',
                              '--lr_step_epochs', '15', '--warmup_epochs', '0', '--grad_clip', '0'],   # upstream never clips; clipped, SGD at 1e-3 barely moves
    'i2s_symsol1_10pct_adamw': ['--optimizer', 'adamw', '--lr', '1e-4', '--lr_scaling', 'sqrt', '--lr_schedule', 'cosine',
                                '--warmup_epochs', '5'],
}
SNAPSHOT_MARKER = 'grid_model_likelihood'   # a repo snapshot / clone must carry the I2S grid log-likelihood
ARGS = COMMON


In [ ]:
import glob, os, shutil, socket, subprocess, tarfile

def online(host='github.com', timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = online()
print('internet:', ONLINE)
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
os.chdir('/kaggle/working')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:        # a repo snapshot dataset, auto-extracted by Kaggle
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:markers[0].index('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
elif tgzs:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
else:
    assert ONLINE, 'no internet and no repo snapshot attached'
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, f'https://github.com/{REPO}.git'],
                       capture_output=True, text=True)
    print(r.stdout + r.stderr); r.check_returncode()
print(subprocess.run(['git', '-C', 'GA-Research', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
src = open('/kaggle/working/GA-Research/pose3d/pose3d/engine/trainer.py').read()
assert SNAPSHOT_MARKER in src, f'stale code: {SNAPSHOT_MARKER!r} is not in engine/trainer.py'


In [ ]:
# Dependencies Kaggle lacks. Online: PyPI / GitHub, with the image's torch pinned by a constraints file.
# Offline: the wheels of the ga-research-offline-deps dataset.
import importlib, importlib.metadata as md, sys
deps = glob.glob('/kaggle/input/**/ga-research-offline-deps', recursive=True)
if deps:
    wheels = sorted(glob.glob(os.path.join(deps[0], '*.whl')))
    print('wheels:', [os.path.basename(w) for w in wheels])
    subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
    ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
    for w in glob.glob('/kaggle/input/**/resnet50-11ad3fa6.pth', recursive=True)[:1]:   # ImageNet weights of resnet50
        shutil.copy(w, os.path.join(ckpt_dir, 'resnet50-11ad3fa6.pth'))
else:
    assert ONLINE, 'no internet and no ga-research-offline-deps attached'
    cons = []
    for p in ['torch', 'torchvision', 'numpy', 'scipy', 'pillow', 'tqdm', 'wandb', 'scikit-image', 'matplotlib',
              'pandas', 'sympy', 'astropy', 'packaging']:
        try:
            cons.append(f'{p}=={md.version(p)}')
        except md.PackageNotFoundError:
            pass
    open('/kaggle/working/constraints.txt', 'w').write('\n'.join(cons) + '\n')
    subprocess.run(['pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', 'e3nn==0.5.9', 'healpy==1.19.0',
                    'git+https://github.com/chagrygoris/image2sphere.git',
                    'git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git'], check=True)
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
print('python', sys.version.split()[0], '| deps import cleanly')


In [ ]:
# W&B: live when online (the wandb_api_key secret, or WANDB_API_KEY above); offline files + WANDB_SYNC lines otherwise.
key = WANDB_API_KEY
if key is None and ONLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('wandb_api_key')
    except Exception as e:   # no secret attached: fall back to offline logging
        print('wandb_api_key secret not available:', e)
if key and ONLINE:
    import wandb
    wandb.login(key=key)
    os.environ['WANDB_API_KEY'] = key
else:
    os.environ['WANDB_MODE'] = 'offline'
print('wandb mode:', os.environ.get('WANDB_MODE', 'online'))


In [ ]:
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
found = glob.glob('/kaggle/input/**/symsol/test/rotations.npz', recursive=True)
assert found, 'attach the SYMSOL dataset (symsol/{train,test}/rotations.npz + images/)'
DATA = found[0][:-len('/symsol/test/rotations.npz')]
print('SYMSOL root:', DATA, flush=True)
failed = []
for name, extra in VARIANTS.items():
    cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}',
           *COMMON, *extra]
    print(' '.join(cmd), flush=True)
    if subprocess.run(cmd, env={**os.environ}).returncode:
        failed.append(name)
assert not failed, f'failed variants: {failed}'   # a crash marks the Kaggle run as failed instead of passing silently
